# Results: leakage-controlled evaluation

Reads `artifacts/metrics.json` and the figures written by `python -m scripts.train`. It does not refit models.

If those files are missing, run training from the repo root first. Numbers and caveats: [`reports/model_card.md`](../reports/model_card.md).



In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
metrics_path = root / "artifacts" / "metrics.json"
if not metrics_path.is_file():
    raise FileNotFoundError(
        f"{metrics_path} not found. From the repo root run:\n"
        "  python -m scripts.train --config configs/default.yaml"
    )
results = json.loads(metrics_path.read_text(encoding="utf-8"))
print(f"methodology={results['methodology']}  seed={results['random_state']}")
print(f"selected_by_validation_auc={results['selected_by_validation_auc']}")



In [ ]:
rows = []
for name, block in results["models"].items():
    test = block["test"]
    rows.append(
        {
            "model": name.replace("_", " "),
            "val_auc": block["val"]["roc_auc"],
            "test_auc": test["roc_auc"],
            "test_ap": test["average_precision"],
            "rej_50": test["rejection_at_50pct_eff"]["background_rejection"],
            "rej_80": test["rejection_at_80pct_eff"]["background_rejection"],
        }
    )
physics = results["physics_baseline"]
rows.append(
    {
        "model": f"physics cut ({physics['feature']})",
        "val_auc": physics["val"]["roc_auc"],
        "test_auc": physics["test"]["roc_auc"],
        "test_ap": physics["test"]["average_precision"],
        "rej_50": physics["test"]["rejection_at_50pct_eff"]["background_rejection"],
        "rej_80": physics["test"]["rejection_at_80pct_eff"]["background_rejection"],
    }
)
table = pd.DataFrame(rows).sort_values("val_auc", ascending=False)
table



## Selected model on the test set

Punzi threshold is frozen from validation. Punzi uses raw counts, not a luminosity-scaled background.



In [ ]:
winner_name = results["selected_by_validation_auc"]
winner = results["models"][winner_name]
punzi = winner["test"]["at_punzi_threshold"]
half = winner["test"]["at_default_threshold"]
display(
    Markdown(
        f"**{winner_name.replace('_', ' ')}**  \n"
        f"Punzi threshold `{punzi['threshold']:.3f}` -> Punzi `{punzi['punzi']:.4f}`, "
        f"precision `{punzi['signal_precision']:.3f}`, recall `{punzi['signal_recall']:.3f}`.  \n"
        f"At 0.5: accuracy `{half['accuracy']:.3f}`, signal F1 `{half['signal_f1']:.3f}`."
    )
)



In [ ]:
figures = root / "artifacts" / "figures"
captions = {
    "roc_test.png": (
        "**Figure 1.** ROC on the held-out test set (natural 35% / 65% mix). "
        "False-positive rate is background efficiency; true-positive rate is signal efficiency."
    ),
    "punzi_validation.png": (
        "**Figure 2.** Punzi significance vs score threshold on validation. "
        "The vertical line is the operating point applied once to test."
    ),
}
for name, caption in captions.items():
    path = figures / name
    if not path.is_file():
        print(f"Missing {path} -- re-run scripts.train to regenerate figures.")
        continue
    display(Image(filename=str(path)))
    display(Markdown(caption))

